# Fake News Prediction Rebuild — Step 4: Vectorize, Split, Train, Compare

Two evaluation upgrades over the old notebook:

1. **Stratified holdout test set carved out first, untouched here.** Reserved for Step 8's final results table, not used for model selection, so that number isn't the same number we tuned against.
2. **TF-IDF fit inside the cross-validation pipeline, not on the whole dataset beforehand.** Vectorizing everything first and splitting after leaks each fold's own vocabulary and IDF weights into what looks like "unseen" validation data. A `Pipeline` inside `cross_validate` refits the vectorizer on each fold's training slice only.

Comparison: title-only vs title+text (the equivalent of the old with/without-author check, since this dataset has no author column). If title+text doesn't clearly beat title-only, that's worth knowing before shipping the bigger feature set.

In [15]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from src.preprocess import build_content

RANDOM_STATE = 42

## Load and prep

Same load/label/shuffle as the EDA notebook, same `random_state`, so row order is identical across notebooks.

In [16]:
# Reload from scratch — content_cache.pkl and the old holdout split
# are both stale now that preprocess.py changed and we're deduping.
fake_df = pd.read_csv('../data/Fake.csv')
true_df = pd.read_csv('../data/True.csv')

fake_df['label'] = 1
true_df['label'] = 0

df = pd.concat([fake_df, true_df], ignore_index=True)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# Dedupe AFTER shuffling, so which copy of a duplicate survives is
# effectively arbitrary rather than biased toward whichever CSV loaded
# first. keep='first' just means "the one that happens to be first
# after the shuffle," not a meaningful choice.
before = len(df)
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
after = len(df)
print(f"Dropped {before - after} duplicate-text rows ({(before - after) / before * 100:.2f}%)")
print(df['label'].value_counts(normalize=True).round(3))

Dropped 6252 duplicate-text rows (13.92%)
label
0    0.548
1    0.452
Name: proportion, dtype: float64


In [18]:
# Rebuild content columns with the updated build_content (now strips
# boilerplate too). No cache check this time on purpose — the whole
# point is these values changed.
df['content_title_only'] = df.apply(
    lambda r: build_content(r['title'], r['text'], include_text=False), axis=1
)
df['content_title_text'] = df.apply(
    lambda r: build_content(r['title'], r['text'], include_text=True), axis=1
)

df.to_pickle('../data/content_cache.pkl')
df[['content_title_only', 'content_title_text']].head(2)

,content_title_only,content_title_text
0,ben stein call th circuit court commit coup ta...,ben stein call th circuit court commit coup ta...
1,trump drop steve bannon nation secur council,trump drop steve bannon nation secur council u...


In [19]:
import re

sample = df[df['label'] == 0].sample(20, random_state=RANDOM_STATE)

for _, row in sample.iterrows():
    cleaned = build_content(row['title'], row['text'], include_text=True)
    if 'reuter' in cleaned or 'edt' in cleaned:
        print("STILL PRESENT:", row.name)
        print(cleaned[:200])
        print('---')

STILL PRESENT: 2146
ex nyc mayor bloomberg say enter presidenti race reuter former new york citi mayor michael bloomberg said monday enter race u presid novemb clear could win want risk enabl victori republican donald tr
---
STILL PRESENT: 11425
u n vote monday call u jerusalem decis withdrawn unit nation secur council due vote monday draft resolut call withdraw u presid donald trump decis recogn jerusalem capit israel diplomat said move like
---
STILL PRESENT: 18670
mexico receiv million catastroph bond quak mexico receiv pay million catastroph bond sept quak met paramet magnitud locat depth financ ministri said statement tuesday reuter report oct govern expect r
---
STILL PRESENT: 12755
senat democrat push new gun control measur reuter lead u senat democrat monday urg quick passag legisl defeat last year impos addit gun control wake weekend mass shoot florida four democrat senat led 
---
STILL PRESENT: 6857
argentina say signal detect like miss submarin hope crew member miss argentin

## Holdout split

15% held out, stratified on label, untouched until Step 8. Everything below trains and compares only on the remaining 85%.

In [ ]:
train_df, holdout_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df['label'],
    random_state=RANDOM_STATE,
)

train_df.to_pickle('../data/train_holdout_split_train.pkl')
holdout_df.to_pickle('../data/train_holdout_split_holdout.pkl')

print(f"train: {train_df.shape}, holdout: {holdout_df.shape}")
print(train_df['label'].value_counts(normalize=True).round(3))
print(holdout_df['label'].value_counts(normalize=True).round(3))

## Cross-validated comparison

5-fold stratified CV, TF-IDF refit inside each fold via `Pipeline`. If this is too slow locally (Random Forest on ~32k documents per fold, 5 folds, 2 variants, 3 models = 30 fits), drop to `n_splits=3` or `RandomForestClassifier(n_estimators=50)` — note it in the README if you do, don't silently change it.

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'LinearSVC': LinearSVC(random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
}

variants = {
    'title_only': 'content_title_only',
    'title_text': 'content_title_text',
}

scoring = ['accuracy', 'f1', 'precision', 'recall']
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

y_train = train_df['label'].values

results = []

for variant_name, col in variants.items():
    X_train = train_df[col].values
    for model_name, model in models.items():
        pipe = Pipeline([
            ('tfidf', TfidfVectorizer()),
            ('clf', model),
        ])
        cv_result = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring, n_jobs=1)
        results.append({
            'variant': variant_name,
            'model': model_name,
            'accuracy_mean': cv_result['test_accuracy'].mean(),
            'accuracy_std': cv_result['test_accuracy'].std(),
            'f1_mean': cv_result['test_f1'].mean(),
            'precision_mean': cv_result['test_precision'].mean(),
            'recall_mean': cv_result['test_recall'].mean(),
        })
        print(f"done: {variant_name} / {model_name}")

results_df = pd.DataFrame(results).round(4)
results_df.sort_values(['variant', 'f1_mean'], ascending=[True, False])

## Read this before picking a model

- Compare `title_only` vs `title_text` rows for the same model. If title+text isn't meaningfully better, that's a real finding, not a failure, say so in the README.
- If any accuracy is above ~98%, don't just take the win. Go back and check: is `content_title_text` accidentally still carrying the Reuters phrase somewhere the regex missed, or a near-duplicate-article problem (wire stories reprinted near-verbatim across rows)? A suspiciously high number after fixing two known leaks is a reason to look for a third, not a reason to celebrate.
- `accuracy_std` across folds tells you how stable the estimate is. A wide std with only 5 folds means the reported mean is noisier than it looks.

Fill in after running:

- Best variant: `<fill in>`
- Best model: `<fill in>`
- Any suspiciously high number investigated: `<fill in>`